In [4]:
import os
import itertools
import numpy as np
import pandas as pd
import random
from discreteNPIV.primal_npJIVE import *
from discreteNPIV.dual_riesz import *
from discreteNPIV.simulate import *
from discreteNPIV.debias_NPIV import *
from discreteNPIV.baselines import *

import os
import itertools
import numpy as np
import pandas as pd
import random
from discreteNPIV.primal_npJIVE import *
from discreteNPIV.dual_riesz import *
from discreteNPIV.simulate import *
from discreteNPIV.debias_NPIV import *
from discreteNPIV.baselines import *

def run_simulation(n, d, K, sparsity, num_simulations=100, seed_setting=None, oracle = True):
    """Runs Monte Carlo simulations for a given setting and returns a dictionary of results."""
    sigma_UX = sigma_UY = sigma_UC = 0.1
    n_new = 10000
    
    # Store results
    plugin_estimates, IPW_estimates, IPW_estimates_no_cross = [], [], []
    debiased_estimates, debiased_estimates_single_fold, debiased_estimates_no_cross = [], [], []
    true_values = []
    ci_coverage = ci_coverage_no_cross = ci_coverage_single_fold = 0
    
    alpha = find_alpha(sparsity, d)
    print(alpha)

    random_seed = seed_setting is None
    for sim in range(num_simulations):
        if random_seed:
            seed_setting = random.randint(0, 2**32 - 1)
        print(f"Running simulation {sim+1}/{num_simulations} for (n={n}, d={d}, K={K}, sparsity={sparsity})")
        
        # Generate data
        data = generate_data(n, K, d, n_new, sigma_UX, sigma_UY, sigma_UC, sparsity, sparsity, seed_setting=seed_setting) 
        dtrain = generate_data(n, K, d, n_new, sigma_UX, sigma_UY, sigma_UC, sparsity, sparsity, seed_setting=seed_setting) 
        
        # Extract data
        theta = data["theta"]
        Y, X, Z, Pi, X_new = data["Y"], data["X"], data["Z"], data["Pi"], data["X_new"]
        true_value = np.mean(X_new @ theta)
        true_values.append(true_value)

        # Estimate using debiased NPIV
        
        out = debias_NPIV(X, Z, Y, X_new, dtrain=dtrain, initial_estimate= true_value if oracle else None, verbose=False)
        out_single_fold = out['output_single_fold']

        # Compute baseline
        out_no_cross = debias_NPIV_no_cross(X, Z, Y, X_new, dtrain=dtrain, lambda_primal=out['lambda_primal'])
        print(true_value, out['estimate'], out_no_cross['estimate'])
        
        plugin_estimates.append(out['estimate_plugin'])
        IPW_estimates.append(out['estimate_IPW'])
        IPW_estimates_no_cross.append(out_no_cross['estimate_IPW'])
        debiased_estimates.append(out['estimate'])
        debiased_estimates_no_cross.append(out_no_cross['estimate'])
        debiased_estimates_single_fold.append(out_single_fold['estimate'])

        # Confidence interval coverage
        if out['ci_lower'] <= true_value <= out['ci_upper']:
            ci_coverage += 1
        if out_no_cross['ci_lower'] <= true_value <= out_no_cross['ci_upper']:
            ci_coverage_no_cross += 1
        if out_single_fold['ci_lower'] <= true_value <= out_single_fold['ci_upper']:
            ci_coverage_single_fold += 1

    # Convert to NumPy arrays
    plugin_estimates = np.array(plugin_estimates)
    IPW_estimates = np.array(IPW_estimates)
    IPW_estimates_no_cross = np.array(IPW_estimates_no_cross)
    debiased_estimates = np.array(debiased_estimates)
    debiased_estimates_no_cross = np.array(debiased_estimates_no_cross)
    debiased_estimates_single_fold = np.array(debiased_estimates_single_fold)
    true_values = np.array(true_values)

    # Compute Bias, Variance, and Mean Squared Error (MSE)
    results = {
        "n": n,
        "d": d,
        "K": K,
        "alpha": alpha,
        "sparsity": sparsity,
        "Bias_Plugin": np.mean(plugin_estimates - true_values),
        "Bias_IPW": np.mean(IPW_estimates - true_values),
        "Bias_IPW_No_Cross": np.mean(IPW_estimates_no_cross - true_values),
        "Bias_Debiased": np.mean(debiased_estimates - true_values),
        "Bias_Debiased_No_Cross": np.mean(debiased_estimates_no_cross - true_values),
        "Bias_Debiased_Single_Fold": np.mean(debiased_estimates_single_fold - true_values),
        "Variance_Plugin": np.var(plugin_estimates - true_values),
        "Variance_IPW": np.var(IPW_estimates - true_values),
        "Variance_IPW_No_Cross": np.var(IPW_estimates_no_cross - true_values),
        "Variance_Debiased": np.var(debiased_estimates - true_values),
        "Variance_Debiased_No_Cross": np.var(debiased_estimates_no_cross - true_values),
        "Variance_Debiased_Single_Fold": np.var(debiased_estimates_single_fold - true_values),
        "MSE_Plugin": np.mean((plugin_estimates - true_values) ** 2),
        "MSE_IPW": np.mean((IPW_estimates - true_values) ** 2),
        "MSE_IPW_No_Cross": np.mean((IPW_estimates_no_cross - true_values) ** 2),
        "MSE_Debiased": np.mean((debiased_estimates - true_values) ** 2),
        "MSE_Debiased_No_Cross": np.mean((debiased_estimates_no_cross - true_values) ** 2),
        "MSE_Debiased_Single_Fold": np.mean((debiased_estimates_single_fold - true_values) ** 2),
        "Coverage_Probability": ci_coverage / num_simulations,
        "Coverage_Probability_No_Cross": ci_coverage_no_cross / num_simulations,
        "Coverage_Probability_Single_Fold": ci_coverage_single_fold / num_simulations
    }

    return results


# Define parameter grid
params = {
    'n': [10], # 10, 50, 100, 200, 
    'd': [200],
    'K_sparsity': [(100, 10), (100, 25), (100, 50), (100, 75), (100, 100), (100, 125), (100, 150), (100, 175), (100, 190), (100, 200)],
    'oracle': [False ]
}

params = {
    'n': [10], # 10, 50, 100, 200, 
    'd': [200],
    'K_sparsity': [(100, 10), (100, 25), (100, 50), (100, 75), (100, 100)],
    'oracle': [True]
}


# Generate all combinations of parameters
param_combinations = list(itertools.product(params['n'], params['d'], params['K_sparsity'], params['oracle']))

# Create results directory
os.makedirs("results", exist_ok=True)

# Run simulations for each parameter setting and save separately
for n, d, (K, sparsity), oracle in param_combinations:
    filename = f"results/sim_n{n}_d{d}_K{K}_sparsity{sparsity}_oracle_{oracle}_final.csv"
    print(filename)
    if os.path.exists(filename):
        continue  # Skip if file exists
    
    result = run_simulation(n, d, K, sparsity, num_simulations = 200, oracle = oracle)

    # Convert to DataFrame
    df_result = pd.DataFrame([result])

    # Define unique filename
    
    # Save results for each setting separately
    df_result.to_csv(filename, index=False)
    
    print(f"Saved results to {filename}")

print("All simulations completed. Results saved separately in the 'results' folder.")


results/sim_n4_d200_K100_sparsity10_oracle_False_final.csv
results/sim_n4_d200_K100_sparsity25_oracle_False_final.csv
results/sim_n4_d200_K100_sparsity50_oracle_False_final.csv
results/sim_n4_d200_K100_sparsity75_oracle_False_final.csv
results/sim_n4_d200_K100_sparsity100_oracle_False_final.csv
results/sim_n4_d200_K100_sparsity125_oracle_False_final.csv
0.49258530139923096
Running simulation 1/200 for (n=4, d=200, K=100, sparsity=125)
1e-05
2.782559402207126e-05
7.742636826811278e-05
0.00021544346900318823
0.0005994842503189409
0.0016681005372000592
0.004641588833612777
0.012915496650148827
0.03593813663804626
0.1
0.1
-0.315188887115798 -0.32359440745845186 -0.2607900062670295
Running simulation 2/200 for (n=4, d=200, K=100, sparsity=125)
1e-05
2.782559402207126e-05
7.742636826811278e-05
0.00021544346900318823
0.0005994842503189409
0.0016681005372000592
0.004641588833612777
0.012915496650148827
0.03593813663804626
0.1
0.1
-3.172495118338634 -2.9357096174332975 -2.903018114741845
Runnin


KeyboardInterrupt



# Testing IPW

In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import random
from discreteNPIV.primal_npJIVE import *
from discreteNPIV.dual_riesz import *
from discreteNPIV.simulate import *
from discreteNPIV.debias_NPIV import *
from discreteNPIV.baselines import *

import os
import itertools
import numpy as np
import pandas as pd
import random
from discreteNPIV.primal_npJIVE import *
from discreteNPIV.dual_riesz import *
from discreteNPIV.simulate import *
from discreteNPIV.debias_NPIV import *
from discreteNPIV.baselines import *

def run_simulation(n, d, K, sparsity, num_simulations=100, seed_setting=None, oracle = True):
    """Runs Monte Carlo simulations for a given setting and returns a dictionary of results."""
    sigma_UX = sigma_UY = sigma_UC = 0.1
    n_new = 10000
    
    # Store results
    plugin_estimates, IPW_estimates, IPW_estimates_no_cross = [], [], []
    debiased_estimates, debiased_estimates_single_fold, debiased_estimates_no_cross = [], [], []
    true_values = []
    ci_coverage = ci_coverage_no_cross = ci_coverage_single_fold = 0
    
    alpha = find_alpha(sparsity, d)
    print(alpha)

    random_seed = seed_setting is None
    for sim in range(num_simulations):
        if random_seed:
            seed_setting = random.randint(0, 2**32 - 1)
        print(f"Running simulation {sim+1}/{num_simulations} for (n={n}, d={d}, K={K}, sparsity={sparsity})")
        
        # Generate data
        data = generate_data(n, K, d, n_new, sigma_UX, sigma_UY, sigma_UC, sparsity, sparsity, seed_setting=seed_setting) 
        dtrain = generate_data(n, K, d, n_new, sigma_UX, sigma_UY, sigma_UC, sparsity, sparsity, seed_setting=seed_setting) 
        
        # Extract data
        theta = data["theta"]
        Y, X, Z, Pi, X_new = data["Y"], data["X"], data["Z"], data["Pi"], data["X_new"]
        true_value = np.mean(X_new @ theta)
        true_values.append(true_value)

        # Estimate using debiased NPIV
        
        out = debias_NPIV(X, Z, Y, X_new, dtrain=dtrain, initial_estimate= true_value if oracle else None, verbose=False)
        out_single_fold = out['output_single_fold']

        # Compute baseline
        out_no_cross = debias_NPIV_no_cross(X, Z, Y, X_new, dtrain=dtrain, lambda_primal=out['lambda_primal'])
        print(true_value, out['estimate'], out_no_cross['estimate'])
        
        plugin_estimates.append(out['estimate_plugin'])
        IPW_estimates.append(out['estimate_IPW'])
        IPW_estimates_no_cross.append(out_no_cross['estimate_IPW'])
        debiased_estimates.append(out['estimate'])
        debiased_estimates_no_cross.append(out_no_cross['estimate'])
        debiased_estimates_single_fold.append(out_single_fold['estimate'])

        # Confidence interval coverage
        if out['ci_lower'] <= true_value <= out['ci_upper']:
            ci_coverage += 1
        if out_no_cross['ci_lower'] <= true_value <= out_no_cross['ci_upper']:
            ci_coverage_no_cross += 1
        if out_single_fold['ci_lower'] <= true_value <= out_single_fold['ci_upper']:
            ci_coverage_single_fold += 1

    # Convert to NumPy arrays
    plugin_estimates = np.array(plugin_estimates)
    IPW_estimates = np.array(IPW_estimates)
    IPW_estimates_no_cross = np.array(IPW_estimates_no_cross)
    debiased_estimates = np.array(debiased_estimates)
    debiased_estimates_no_cross = np.array(debiased_estimates_no_cross)
    debiased_estimates_single_fold = np.array(debiased_estimates_single_fold)
    true_values = np.array(true_values)

    # Compute Bias, Variance, and Mean Squared Error (MSE)
    results = {
        "n": n,
        "d": d,
        "K": K,
        "alpha": alpha,
        "sparsity": sparsity,
        "Bias_Plugin": np.mean(plugin_estimates - true_values),
        "Bias_IPW": np.mean(IPW_estimates - true_values),
        "Bias_IPW_No_Cross": np.mean(IPW_estimates_no_cross - true_values),
        "Bias_Debiased": np.mean(debiased_estimates - true_values),
        "Bias_Debiased_No_Cross": np.mean(debiased_estimates_no_cross - true_values),
        "Bias_Debiased_Single_Fold": np.mean(debiased_estimates_single_fold - true_values),
        "Variance_Plugin": np.var(plugin_estimates - true_values),
        "Variance_IPW": np.var(IPW_estimates - true_values),
        "Variance_IPW_No_Cross": np.var(IPW_estimates_no_cross - true_values),
        "Variance_Debiased": np.var(debiased_estimates - true_values),
        "Variance_Debiased_No_Cross": np.var(debiased_estimates_no_cross - true_values),
        "Variance_Debiased_Single_Fold": np.var(debiased_estimates_single_fold - true_values),
        "MSE_Plugin": np.mean((plugin_estimates - true_values) ** 2),
        "MSE_IPW": np.mean((IPW_estimates - true_values) ** 2),
        "MSE_IPW_No_Cross": np.mean((IPW_estimates_no_cross - true_values) ** 2),
        "MSE_Debiased": np.mean((debiased_estimates - true_values) ** 2),
        "MSE_Debiased_No_Cross": np.mean((debiased_estimates_no_cross - true_values) ** 2),
        "MSE_Debiased_Single_Fold": np.mean((debiased_estimates_single_fold - true_values) ** 2),
        "Coverage_Probability": ci_coverage / num_simulations,
        "Coverage_Probability_No_Cross": ci_coverage_no_cross / num_simulations,
        "Coverage_Probability_Single_Fold": ci_coverage_single_fold / num_simulations
    }

    return results


# Define parameter grid
params = {
    'n': [10], # 10, 50, 100, 200, 
    'd': [200],
    'K_sparsity': [(100, 10), (100, 25), (100, 50), (100, 75), (100, 100), (100, 125), (100, 150), (100, 175), (100, 190), (100, 200)],
    'oracle': [False ]
}

params = {
    'n': [10], # 10, 50, 100, 200, 
    'd': [200],
    'K_sparsity': [(100, 10), (100, 25), (100, 50), (100, 75), (100, 100)],
    'oracle': [True]
}


# Generate all combinations of parameters
param_combinations = list(itertools.product(params['n'], params['d'], params['K_sparsity'], params['oracle']))

# Create results directory
os.makedirs("results", exist_ok=True)

# Run simulations for each parameter setting and save separately
for n, d, (K, sparsity), oracle in param_combinations:
    filename = f"results/sim_n{n}_d{d}_K{K}_sparsity{sparsity}_oracle_{oracle}_final.csv"
    print(filename)
    if os.path.exists(filename):
        continue  # Skip if file exists
    
    result = run_simulation(n, d, K, sparsity, num_simulations = 200, oracle = oracle)

    # Convert to DataFrame
    df_result = pd.DataFrame([result])

    # Define unique filename
    
    # Save results for each setting separately
    df_result.to_csv(filename, index=False)
    
    print(f"Saved results to {filename}")

print("All simulations completed. Results saved separately in the 'results' folder.")


# old 

In [ ]:
import numpy as np
from discreteNPIV.simulate import *

# Parameters
d = 200
target_sparsity = 50

# Find alpha that achieves the target effective sparsity
alpha = find_alpha(target_sparsity, d)
j_vals = np.arange(1, d + 1)
decay = j_vals ** (-alpha)

# Compute truncation error
s = target_sparsity  # Assuming truncation at the effective sparsity level
error = truncation_error(decay, target_sparsity)

# Output
print("Found alpha:", alpha)
print("Effective sparsity:", effective_sparsity(alpha, j_vals))
print("Relative l^2 truncation error:", error)


In [ ]:
from discreteNPIV.primal_npJIVE import *
from discreteNPIV.dual_riesz import *
from discreteNPIV.simulate import *
from discreteNPIV.debias_NPIV import *
import numpy as np

params = {
    'n': [10, 20, 30, 50, 100, 500, 1000],
    'd': [200],
    'K_sparsity': [(100, 10), (100, 30), (100, 50), (100, 75), (100, 100), (100, 200)]
}
    
# Simulation settings
n = 20  # Number of individuals
d = 200
K = 100
sparsity = 50
sigma_UX = 0.1
sigma_UY = 0.1
sigma_UC = 0.1
n_new = 10000
num_simulations = 40  # Number of Monte Carlo simulations
seed_setting = 235
# Store results
plugin_estimates = []
IPW_estimates = []
debiased_estimates = []
true_values = []
ci_coverage = 0
 
for sim in range(num_simulations):
    print(sim)
    # Generate data
    data = generate_data(n, K, d, n_new, sigma_UX, sigma_UY, sigma_UC, sparsity, sparsity, seed_setting=seed_setting) 
    dtrain = generate_data(n, K, d, n_new, sigma_UX, sigma_UY, sigma_UC, sparsity, sparsity, seed_setting=seed_setting) 
    
    # Unpack dictionary
    theta = data["theta"]  # Sparse coefficient vector
    Y = data["Y"]  # Flattened response variable
    X = data["X"]  # Flattened covariates matrix
    Z = data["Z"]  # Group indicator for each row
    Pi = data["Pi"]  # Group-level feature means
    X_new = data["X_new"]  # New covariates sampled
    
    true_value = np.mean(X_new @ theta)  # True mean
    true_values.append(true_value)

    # Estimate using debiased NPIV
    out = debias_NPIV(X, Z, Y, X_new, dtrain=dtrain, verbose=False)
    
    estimate_plugin = out['estimate_plugin']
    estimate_IPW = out['estimate_IPW']
    estimate_debiased = out['estimate']
    ci_lower = out['ci_lower']
    ci_upper = out['ci_upper']
    
    # Store estimates
    plugin_estimates.append(estimate_plugin)
    IPW_estimates.append(estimate_IPW)
    debiased_estimates.append(estimate_debiased)
    
    # Coverage check
    if ci_lower <= true_value <= ci_upper:
        ci_coverage += 1

# Convert to NumPy arrays
plugin_estimates = np.array(plugin_estimates)
IPW_estimates = np.array(IPW_estimates)
debiased_estimates = np.array(debiased_estimates)
true_values = np.array(true_values)

# Compute Bias, Variance, and Mean Squared Error (MSE)
bias_plugin = np.mean(plugin_estimates - true_values)
bias_IPW = np.mean(IPW_estimates - true_values)
bias_debiased = np.mean(debiased_estimates - true_values)

var_plugin = np.var(plugin_estimates)
var_IPW = np.var(IPW_estimates)
var_debiased = np.var(debiased_estimates)

mse_plugin = np.mean((plugin_estimates - true_values) ** 2)
mse_IPW = np.mean((IPW_estimates - true_values) ** 2)
mse_debiased = np.mean((debiased_estimates - true_values) ** 2)

coverage_prob = ci_coverage / num_simulations  # Should be ~0.95

# Print results
print(f"Bias (Plugin): {bias_plugin:.4f}, Variance: {var_plugin:.4f}, MSE: {mse_plugin:.4f}")
print(f"Bias (IPW): {bias_IPW:.4f}, Variance: {var_IPW:.4f}, MSE: {mse_IPW:.4f}")
print(f"Bias (Debiased): {bias_debiased:.4f}, Variance: {var_debiased:.4f}, MSE: {mse_debiased:.4f}")
print(f"Coverage Probability (Debiased, 95% CI): {coverage_prob:.4f}")



In [ ]:
import numpy as np

# Create cross-validation folds for training and evaluation
cross_folds_train = make_cross_folds(dtrain['Z'])
cross_folds = make_cross_folds(Z)

# Compute group proportions (W) for normalization
n_total = len(Z)
group_counts = np.bincount(Z, minlength=K)
W = group_counts / n_total  # Shape (K,)

# Perform initial fit using JIVE (nonparametric Joint IV Estimation)
best_lambda, cross_folds = cross_validate_JIVE(
    dtrain['X'], dtrain['Z'], dtrain['Y'], K, cross_folds=cross_folds_train
)
theta_hat, h_fun_init = solve_primal_npJIVE(
    dtrain['X'], dtrain['Z'], dtrain['Y'], K, cross_folds=cross_folds_train, lambda_K=best_lambda
)
h_hat = h_fun_init(X)  # Compute fitted values

# Perform linear calibration of the function h
theta_hat_cal, h_fun_cal = solve_primal_npJIVE(
    h_hat.reshape(-1, 1), Z, Y, K, cross_folds=cross_folds, lambda_K=1e-10
)

def h_fun(X):
    """Calibrated function h using sequential estimation."""
    return h_fun_cal(h_fun_init(X).reshape(-1, 1))
h_hat = h_fun(X)  # Update h_hat using calibrated function

# Perform cross-validation for the dual Riesz estimator
best_lambda, _ = cross_validate_dual(
    dtrain['X'], dtrain['Z'], K, X_new, cross_folds=cross_folds_train
)

# Fit dual Riesz representation (b_hat, beta_fun_init)
b_hat, beta_fun_init = solve_dual_riesz(
    dtrain['X'], dtrain['Z'], K, X_new, cross_folds=cross_folds_train, lambda_K=1e-8
)
beta_hat = beta_fun_init(X)  # Compute fitted values

# Compute stabilization factor
T_beta_hat_0 = compute_group_means(beta_hat, Z, cross_folds, 0, K)
T_beta_hat_1 = compute_group_means(beta_hat, Z, cross_folds, 1, K)
stabilize = np.mean(beta_fun_init(X_new)) / np.sum(W * T_beta_hat_0 * T_beta_hat_1)
print("Stable (before):", stabilize)
estimates_IPW =   np.average(
    T_beta_hat_0 * compute_group_means(Y, Z, cross_folds, 1, K), weights=W
) 
print("IPW:", estimates_IPW)


# Perform linear calibration of the function beta
b_hat_cal, beta_fun_cal = solve_dual_riesz(
    beta_hat.reshape(-1, 1), Z, K, beta_fun_init(X_new).reshape(-1, 1), cross_folds=cross_folds, lambda_K=1e-10
)

def beta_fun(X):
    """Calibrated function beta using sequential estimation."""
    return beta_fun_cal(beta_fun_init(X).reshape(-1, 1))

beta_hat = beta_fun(X)  # Update beta_hat using calibrated function

# Compute group means for stabilized importance weighting
T_beta_hat_0 = compute_group_means(beta_hat, Z, cross_folds, 0, K)
T_beta_hat_1 = compute_group_means(beta_hat, Z, cross_folds, 1, K)
T_Y_1 = compute_group_means(Y, Z, cross_folds, 1, K)
T_Y_0 = compute_group_means(Y, Z, cross_folds, 0, K)
T_h_hat_0 = compute_group_means(h_hat, Z, cross_folds, 0, K)
T_h_hat_1 = compute_group_means(h_hat, Z, cross_folds, 1, K)
beta_new = beta_fun(X_new)
h_new = h_fun(X_new)

EIF = ((cross_folds == 1) * T_beta_hat_0[Z] * (Y - h_hat) + (cross_folds == 0) * T_beta_hat_1[Z] * (Y - h_hat))
print("plug-in bias: ", np.mean(EIF))
print("IPW bias: ", np.sum(0.5*(W * (T_beta_hat_0 * T_h_hat_1 +  T_beta_hat_1 * T_h_hat_0))) - np.mean(h_new))
# Compute stabilization factor
stabilize = np.mean(beta_new) / np.sum(W * T_beta_hat_0 * T_beta_hat_1)
print("Stable (after):", stabilize)

se = np.std(EIF) / np.sqrt(len(EIF))
print(se)
# Compute different estimators
estimate_plugin = np.mean(h_new)  # Plugin estimator
estimates_IPW = 0.5 * (np.sum(W * T_beta_hat_0 * T_Y_1) +  np.sum(W * T_beta_hat_1 * T_Y_0))  # Inverse probability weighting estimator
estimate_debiased = estimate_plugin + 0.5 * (np.sum(W * T_beta_hat_0 * (T_Y_1 - T_h_hat_1)) +  np.sum(W * T_beta_hat_1 * (T_Y_0 - T_h_hat_0)))

# Output the estimates
 
print(estimate_plugin, estimate_debiased, estimates_IPW, true)
print(estimate_debiased, estimate_debiased - 1.96 * se, estimate_debiased + 1.96 * se, true)


In [ ]:
print(np.average(
    T_beta_hat_0 * T_Y_1, weights=W
) )
print(np.average(
    T_beta_hat_1 * T_Y_0, weights=W
) )

In [ ]:
min(Z)